# GTF-based end-to-end subisoform simulation

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "tealeaf").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)


In [ ]:
from pathlib import Path

import pandas as pd

from tealeaf.subisoform_simulation import (
    build_gtf_event_simulation_design,
    run_end_to_end_simulation,
)


In [ ]:
gtf_path = Path("/gpfs/commons/groups/knowles_lab/index/kallisto/mus_musculus/with_precursor/gencode.vM32.basic.annotation.gtf")
transcript_ids = ["ENSMUST00000204394.3", "ENSMUST00000204423.3"]
design = build_gtf_event_simulation_design(gtf_path, transcript_ids=transcript_ids)

design.subisoform_model.subisoform_table.loc[
    design.subisoform_model.subisoform_table.event_id == design.target_event_id,
    ["event_id", "subisoform_id", "transcript_ids", "segment_ids"],
]

In [ ]:
simulation, pipeline = run_end_to_end_simulation(
    design=design,
    alpha_by_condition={"a": [18, 2], "b": [2, 18]},
    n_cells_per_condition=18,
    mean_gene_count=120,
    metacell_size=3,
    random_state=0,
)

pd.DataFrame(
    {
        "condition": simulation.group_labels,
        "total_ec_count": simulation.cell_ec_matrix.sum(axis=1).A1,
    }
).groupby("condition").agg(["size", "mean"])

In [ ]:
pipeline.differential_usage.loc[:, ["event_id", "n_subisoforms", "n_samples", "lrt_statistic", "p_value", "fdr"]]